In [ ]:
# ================================================================
# GPU RANDOM FINITE-AUTOMATA SAT SEARCH
# Google Colab — ONE CELL
#
# Thousands of random finite automata run in parallel on CUDA.
#
# Upload DIMACS .cnf
# -> GPU batch of random automata
# -> Each automaton controls one SAT walker
# -> Periodic updates only
# -> On SAT: CPU-verifies witness
# -> Downloads SAT_solution.txt
# ================================================================

from google.colab import files
from IPython.display import clear_output

import torch
import random
import time
import os
import math


# ================================================================
# CONFIG
# ================================================================

# Number of finite states per random automaton
Q = 32

# Parallel automata / SAT walkers on GPU.
#
# T4: start with 2048 or 4096
# L4/A100: try 8192, 16384+
BATCH_AUTOMATA = 4096

# Steps performed before all automata are discarded and regenerated
STEPS_PER_GENERATION = 20000

# Process clauses in chunks to avoid BATCH x ALL_CLAUSES memory explosion
CLAUSE_CHUNK = 512

# Only refresh screen every few seconds
UPDATE_SECONDS = 3.0

# Random exploration probability
NOISE = 0.08

# Continue forever until SAT.
# Set integer if you want a limit.
MAX_GENERATIONS = None

MASTER_SEED = int.from_bytes(os.urandom(8), "little")


# ================================================================
# CUDA CHECK
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected.\n"
        "In Colab: Runtime -> Change runtime type -> GPU"
    )

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


# ================================================================
# UPLOAD
# ================================================================

print("\nUpload DIMACS .cnf ...")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

filename = next(iter(uploaded))
text = uploaded[filename].decode("utf-8", errors="ignore")

clear_output(wait=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Loaded:", filename)
print("Parsing...")


# ================================================================
# DIMACS PARSER
# ================================================================

def parse_dimacs(text):

    clauses = []
    current = []
    declared_vars = 0

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        if line.startswith("c"):
            continue

        if line.startswith("%"):
            break

        if line.startswith("p"):

            p = line.split()

            if len(p) >= 4:
                declared_vars = int(p[2])

            continue

        for token in line.split():

            try:
                x = int(token)
            except:
                continue

            if x == 0:

                s = set(current)

                # Remove tautological clauses
                if not any(-lit in s for lit in s):
                    clauses.append(list(s))

                current = []

            else:
                current.append(x)

    if current:

        s = set(current)

        if not any(-lit in s for lit in s):
            clauses.append(list(s))

    inferred = max(
        (
            abs(l)
            for c in clauses
            for l in c
        ),
        default=0
    )

    return max(declared_vars, inferred), clauses


N, CLAUSES = parse_dimacs(text)
M = len(CLAUSES)

if N == 0:
    raise RuntimeError("No variables detected.")

if any(len(c) == 0 for c in CLAUSES):
    raise RuntimeError("CNF contains empty clause => UNSAT.")

MAX_WIDTH = max(len(c) for c in CLAUSES)

print(f"Variables: {N:,}")
print(f"Clauses:   {M:,}")
print(f"Max width: {MAX_WIDTH}")


# ================================================================
# GPU CLAUSE REPRESENTATION
#
# vars_gpu[c,j] = variable index
# signs_gpu[c,j] = desired truth value
# mask_gpu[c,j] = literal exists
# ================================================================

vars_pad = torch.zeros(
    (M, MAX_WIDTH),
    dtype=torch.long
)

sign_pad = torch.zeros(
    (M, MAX_WIDTH),
    dtype=torch.bool
)

mask_pad = torch.zeros(
    (M, MAX_WIDTH),
    dtype=torch.bool
)

lengths = torch.zeros(
    M,
    dtype=torch.long
)


for ci, clause in enumerate(CLAUSES):

    lengths[ci] = len(clause)

    for j, lit in enumerate(clause):

        vars_pad[ci, j] = abs(lit) - 1
        sign_pad[ci, j] = lit > 0
        mask_pad[ci, j] = True


vars_gpu = vars_pad.to(device)
signs_gpu = sign_pad.to(device)
mask_gpu = mask_pad.to(device)
lengths_gpu = lengths.to(device)


# Static variable occurrence count
occurrence = torch.zeros(N, dtype=torch.long)

for clause in CLAUSES:
    for lit in clause:
        occurrence[abs(lit) - 1] += 1

occurrence_gpu = occurrence.to(device)


# ================================================================
# CPU VERIFIER
# ================================================================

def verify_cpu(assignment):

    for clause in CLAUSES:

        ok = False

        for lit in clause:

            value = assignment[abs(lit) - 1]

            if lit < 0:
                value = not value

            if value:
                ok = True
                break

        if not ok:
            return False

    return True


# ================================================================
# RANDOM AUTOMATA
#
# Each GPU walker owns:
#
# delta[walker, q, observation]
#
# observation:
#   0 = unsatisfied clauses decreased
#   1 = same
#   2 = increased
#
# action[walker, q]:
#
#   0 = random literal
#   1 = automaton slot
#   2 = high-occurrence variable
#   3 = random tournament
# ================================================================

def generate_automata(batch):

    delta = torch.randint(
        0,
        Q,
        (batch, Q, 3),
        device=device,
        dtype=torch.long
    )

    actions = torch.randint(
        0,
        4,
        (batch, Q),
        device=device,
        dtype=torch.long
    )

    salts = torch.randint(
        0,
        2**31 - 1,
        (batch, Q),
        device=device,
        dtype=torch.long
    )

    states = torch.randint(
        0,
        Q,
        (batch,),
        device=device,
        dtype=torch.long
    )

    return delta, actions, salts, states


# ================================================================
# RANDOM ASSIGNMENTS
# ================================================================

def random_assignments(batch):

    return torch.randint(
        0,
        2,
        (batch, N),
        device=device,
        dtype=torch.bool
    )


# ================================================================
# EVALUATE CNF
#
# Returns:
#
# unsat_count[walker]
# selected_unsat_clause[walker]
#
# Clause selection is randomized with GPU random scores.
# ================================================================

@torch.no_grad()
def evaluate(assignments):

    B = assignments.shape[0]

    unsat_count = torch.zeros(
        B,
        device=device,
        dtype=torch.long
    )

    best_score = torch.full(
        (B,),
        -1.0,
        device=device
    )

    chosen_clause = torch.zeros(
        B,
        device=device,
        dtype=torch.long
    )

    rows = torch.arange(
        B,
        device=device
    )[:, None, None]

    for start in range(0, M, CLAUSE_CHUNK):

        end = min(
            start + CLAUSE_CHUNK,
            M
        )

        v = vars_gpu[start:end]
        s = signs_gpu[start:end]
        mk = mask_gpu[start:end]

        # B x chunk x width
        values = assignments[:, v]

        literal_true = (
            values == s
        ) & mk

        clause_sat = literal_true.any(
            dim=2
        )

        unsat = ~clause_sat

        unsat_count += unsat.sum(
            dim=1
        )

        # Random key for each unsatisfied clause
        scores = torch.rand(
            (B, end - start),
            device=device
        )

        scores = torch.where(
            unsat,
            scores,
            torch.full_like(
                scores,
                -1.0
            )
        )

        local_score, local_idx = scores.max(
            dim=1
        )

        replace = (
            local_score > best_score
        )

        best_score = torch.where(
            replace,
            local_score,
            best_score
        )

        chosen_clause = torch.where(
            replace,
            local_idx + start,
            chosen_clause
        )

    return unsat_count, chosen_clause


# ================================================================
# SELECT VARIABLES FROM UNSAT CLAUSES
# ================================================================

@torch.no_grad()
def choose_variables(
    chosen_clause,
    controller_state,
    actions,
    salts
):

    B = chosen_clause.shape[0]

    row = torch.arange(
        B,
        device=device
    )

    clause_vars = vars_gpu[
        chosen_clause
    ]

    clause_len = lengths_gpu[
        chosen_clause
    ]

    action = actions[
        row,
        controller_state
    ]

    salt = salts[
        row,
        controller_state
    ]

    # ------------------------------------------------------------
    # Action 0:
    # random literal
    # ------------------------------------------------------------

    rand_slot = (
        torch.rand(
            B,
            device=device
        )
        * clause_len.float()
    ).long()

    variable_random = clause_vars[
        row,
        rand_slot
    ]


    # ------------------------------------------------------------
    # Action 1:
    # deterministic automaton-selected slot
    # ------------------------------------------------------------

    fixed_slot = salt % clause_len

    variable_fixed = clause_vars[
        row,
        fixed_slot
    ]


    # ------------------------------------------------------------
    # Action 2:
    # choose variable with highest static occurrence
    # ------------------------------------------------------------

    occ = occurrence_gpu[
        clause_vars
    ]

    valid = (
        torch.arange(
            MAX_WIDTH,
            device=device
        )[None, :]
        < clause_len[:, None]
    )

    occ = torch.where(
        valid,
        occ,
        torch.full_like(
            occ,
            -1
        )
    )

    high_slot = occ.argmax(
        dim=1
    )

    variable_high = clause_vars[
        row,
        high_slot
    ]


    # ------------------------------------------------------------
    # Action 3:
    # two random literal tournament
    #
    # prefers lower-occurrence candidate
    # ------------------------------------------------------------

    slot_a = (
        torch.rand(
            B,
            device=device
        )
        * clause_len.float()
    ).long()

    slot_b = (
        torch.rand(
            B,
            device=device
        )
        * clause_len.float()
    ).long()

    var_a = clause_vars[
        row,
        slot_a
    ]

    var_b = clause_vars[
        row,
        slot_b
    ]

    choose_a = (
        occurrence_gpu[var_a]
        <=
        occurrence_gpu[var_b]
    )

    variable_tournament = torch.where(
        choose_a,
        var_a,
        var_b
    )


    # ------------------------------------------------------------
    # Combine automaton actions
    # ------------------------------------------------------------

    variable = torch.where(
        action == 0,
        variable_random,
        variable_fixed
    )

    variable = torch.where(
        action == 2,
        variable_high,
        variable
    )

    variable = torch.where(
        action == 3,
        variable_tournament,
        variable
    )


    # ------------------------------------------------------------
    # WalkSAT-style noise
    # ------------------------------------------------------------

    noise_mask = (
        torch.rand(
            B,
            device=device
        )
        < NOISE
    )

    variable = torch.where(
        noise_mask,
        variable_random,
        variable
    )

    return variable


# ================================================================
# OUTPUT
# ================================================================

def save_solution(
    assignment,
    generation,
    walker,
    step,
    elapsed,
    delta,
    actions,
    controller_state
):

    path = "/content/SAT_solution.txt"

    literals = [
        str(i + 1)
        if assignment[i]
        else str(-(i + 1))
        for i in range(N)
    ]

    action_names = [
        "random-literal",
        "automaton-slot",
        "high-occurrence",
        "tournament"
    ]

    with open(path, "w") as f:

        f.write(
            "c GPU Random Finite-Automata SAT Search\n"
        )

        f.write(
            f"c source {filename}\n"
        )

        f.write(
            f"c gpu {torch.cuda.get_device_name(0)}\n"
        )

        f.write(
            f"c variables {N}\n"
        )

        f.write(
            f"c clauses {M}\n"
        )

        f.write(
            f"c generation {generation}\n"
        )

        f.write(
            f"c winning_gpu_walker {walker}\n"
        )

        f.write(
            f"c winning_step {step}\n"
        )

        f.write(
            f"c elapsed_seconds {elapsed:.6f}\n"
        )

        f.write(
            "c witness verified independently on CPU\n\n"
        )

        f.write(
            "s SATISFIABLE\n"
        )

        chunk = 30

        for i in range(
            0,
            N,
            chunk
        ):

            part = literals[
                i:i + chunk
            ]

            suffix = (
                " 0"
                if i + chunk >= N
                else ""
            )

            f.write(
                "v "
                + " ".join(part)
                + suffix
                + "\n"
            )


        # --------------------------------------------------------
        # Winning finite automaton
        # --------------------------------------------------------

        f.write(
            "\nc WINNING FINITE AUTOMATON\n"
        )

        f.write(
            f"c current_state q{controller_state}\n"
        )

        for q in range(Q):

            f.write(
                f"c q{q}: "
                f"better->q{delta[q][0]} "
                f"same->q{delta[q][1]} "
                f"worse->q{delta[q][2]} "
                f"action={action_names[actions[q]]}\n"
            )

    return path


# ================================================================
# STATUS PANEL
# ================================================================

start_time = time.time()
last_update = 0

generation = 0
total_gpu_walkers = 0
total_steps = 0

global_best = M


def status(
    current_best,
    step,
    force=False
):

    global last_update

    now = time.time()

    if (
        not force
        and now - last_update
        < UPDATE_SECONDS
    ):
        return

    last_update = now

    elapsed = (
        now - start_time
    )

    walker_steps_sec = (
        total_steps / elapsed
        if elapsed > 0
        else 0
    )

    clear_output(wait=True)

    gpu_mem = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    print(
        "╔════════════════════════════════════════════════════╗"
    )

    print(
        "║       GPU RANDOM FINITE-AUTOMATA SAT SEARCH        ║"
    )

    print(
        "╚════════════════════════════════════════════════════╝"
    )

    print()

    print(
        f"GPU:                  {torch.cuda.get_device_name(0)}"
    )

    print(
        f"CNF:                  {filename}"
    )

    print(
        f"Variables:            {N:,}"
    )

    print(
        f"Clauses:              {M:,}"
    )

    print()

    print(
        f"Parallel automata:    {BATCH_AUTOMATA:,}"
    )

    print(
        f"States / automaton:   {Q}"
    )

    print(
        f"Generation:           {generation:,}"
    )

    print(
        f"Step:                 {step:,}/{STEPS_PER_GENERATION:,}"
    )

    print()

    print(
        f"Current best unsat:   {current_best:,}"
    )

    print(
        f"Global best unsat:    {global_best:,}"
    )

    print()

    print(
        f"Automata launched:    {total_gpu_walkers:,}"
    )

    print(
        f"Walker-steps:         {total_steps:,}"
    )

    print(
        f"Walker-steps/sec:     {walker_steps_sec:,.0f}"
    )

    print()

    print(
        f"GPU memory used:      {gpu_mem:.2f} GB"
    )

    print(
        f"Elapsed:              {elapsed:.1f} s"
    )

    print()

    print(
        "CUDA search running..."
    )


# ================================================================
# MASTER SEARCH
# ================================================================

torch.manual_seed(
    MASTER_SEED % (2**63 - 1)
)

random.seed(
    MASTER_SEED
)

solution = None

status(M, 0, force=True)


try:

    while True:

        generation += 1

        if (
            MAX_GENERATIONS is not None
            and generation > MAX_GENERATIONS
        ):
            break

        # --------------------------------------------------------
        # Entire new population of finite automata
        # --------------------------------------------------------

        delta, actions, salts, controller_state = (
            generate_automata(
                BATCH_AUTOMATA
            )
        )

        assignments = random_assignments(
            BATCH_AUTOMATA
        )

        previous_unsat, chosen_clause = evaluate(
            assignments
        )

        total_gpu_walkers += (
            BATCH_AUTOMATA
        )


        # Immediate SAT?
        sat_rows = torch.nonzero(
            previous_unsat == 0,
            as_tuple=False
        ).flatten()

        if sat_rows.numel():

            winner = int(
                sat_rows[0].item()
            )

            solution = (
                assignments[winner]
                .detach()
                .cpu()
                .tolist(),
                generation,
                winner,
                0,
                delta[winner]
                .detach()
                .cpu()
                .tolist(),
                actions[winner]
                .detach()
                .cpu()
                .tolist(),
                int(
                    controller_state[winner]
                    .item()
                )
            )

            break


        # --------------------------------------------------------
        # GPU search
        # --------------------------------------------------------

        row = torch.arange(
            BATCH_AUTOMATA,
            device=device
        )

        for step in range(
            1,
            STEPS_PER_GENERATION + 1
        ):

            variable = choose_variables(
                chosen_clause,
                controller_state,
                actions,
                salts
            )

            # Flip one Boolean variable per GPU automaton
            assignments[
                row,
                variable
            ] = ~assignments[
                row,
                variable
            ]

            new_unsat, chosen_clause = evaluate(
                assignments
            )

            # ----------------------------------------------------
            # Feed search result back into automaton
            #
            # 0 = better
            # 1 = same
            # 2 = worse
            # ----------------------------------------------------

            observation = torch.ones(
                BATCH_AUTOMATA,
                device=device,
                dtype=torch.long
            )

            observation = torch.where(
                new_unsat
                < previous_unsat,
                torch.zeros_like(
                    observation
                ),
                observation
            )

            observation = torch.where(
                new_unsat
                > previous_unsat,
                torch.full_like(
                    observation,
                    2
                ),
                observation
            )

            controller_state = delta[
                row,
                controller_state,
                observation
            ]

            previous_unsat = new_unsat

            total_steps += (
                BATCH_AUTOMATA
            )

            current_best = int(
                new_unsat.min().item()
            )

            if current_best < global_best:
                global_best = current_best


            # ----------------------------------------------------
            # SAT FOUND
            # ----------------------------------------------------

            sat_rows = torch.nonzero(
                new_unsat == 0,
                as_tuple=False
            ).flatten()

            if sat_rows.numel():

                winner = int(
                    sat_rows[0].item()
                )

                solution = (
                    assignments[winner]
                    .detach()
                    .cpu()
                    .tolist(),

                    generation,

                    winner,

                    step,

                    delta[winner]
                    .detach()
                    .cpu()
                    .tolist(),

                    actions[winner]
                    .detach()
                    .cpu()
                    .tolist(),

                    int(
                        controller_state[winner]
                        .item()
                    )
                )

                break


            status(
                current_best,
                step
            )


        if solution is not None:
            break


except KeyboardInterrupt:

    clear_output(wait=True)

    print(
        "Search stopped manually."
    )

    print(
        "Best unsatisfied clauses:",
        global_best
    )


# ================================================================
# VERIFY + DOWNLOAD
# ================================================================

if solution is not None:

    (
        assignment,
        winning_generation,
        winning_walker,
        winning_step,
        winning_delta,
        winning_actions,
        winning_state
    ) = solution


    clear_output(wait=True)

    print(
        "Potential SAT witness found on GPU."
    )

    print(
        "Running independent CPU verification..."
    )

    verified = verify_cpu(
        assignment
    )

    if not verified:

        raise RuntimeError(
            "GPU candidate failed CPU verification."
        )


    elapsed = (
        time.time()
        - start_time
    )


    path = save_solution(

        assignment,

        winning_generation,

        winning_walker,

        winning_step,

        elapsed,

        winning_delta,

        winning_actions,

        winning_state
    )


    clear_output(wait=True)

    print(
        "╔════════════════════════════════════════════════════╗"
    )

    print(
        "║                    SAT FOUND                       ║"
    )

    print(
        "╚════════════════════════════════════════════════════╝"
    )

    print()

    print(
        f"GPU:                {torch.cuda.get_device_name(0)}"
    )

    print(
        f"Generation:         {winning_generation:,}"
    )

    print(
        f"Winning walker:     {winning_walker:,}"
    )

    print(
        f"Winning step:       {winning_step:,}"
    )

    print(
        f"Parallel automata:  {BATCH_AUTOMATA:,}"
    )

    print(
        f"Total walker-steps: {total_steps:,}"
    )

    print(
        f"Elapsed:            {elapsed:.3f} s"
    )

    print()

    print(
        "CPU witness verification: PASS ✓"
    )

    print()

    preview = []

    for i in range(
        min(20, N)
    ):

        preview.append(
            f"x{i+1}="
            + (
                "1"
                if assignment[i]
                else "0"
            )
        )

    print(
        "Witness preview:"
    )

    print(
        " ".join(preview)
    )

    if N > 20:
        print("...")

    print()

    print(
        "Winner automaton saved inside SAT_solution.txt"
    )

    print(
        path
    )

    files.download(path)


elif MAX_GENERATIONS is not None:

    clear_output(wait=True)

    print(
        "Configured generation limit reached."
    )

    print(
        f"Best unsatisfied clauses: {global_best}"
    )

    print()

    print(
        "No SAT witness found."
    )

    print(
        "This does not establish UNSAT."
    )

GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.8

Upload DIMACS .cnf ...
